<a href="https://colab.research.google.com/github/StathisDevves/Industrial/blob/main/Minimill%20Lowest%20Prices%20Overall%20Opt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q pyomo pandas openpyxl
!apt-get update -qq
!apt-get install -y -qq glpk-utils

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [6]:
from google.colab import files
uploaded = files.upload()

Saving Electricity_Gas Solar Prices_2025_8760.xlsx to Electricity_Gas Solar Prices_2025_8760.xlsx


In [7]:
import os
import pandas as pd
import numpy as np
from pyomo.environ import (
    ConcreteModel, Var, Binary, NonNegativeReals, RangeSet,
    Constraint, Objective, minimize, SolverFactory, value
)
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

In [43]:
INPUT_FILE = "Electricity_Gas Solar Prices_2025_8760.xlsx"
OUTPUT_FILE = "MiniMill_Design_Formula_Integrated_Output_GLPK.xlsx"

# You clarified that:
# - first choose 8 consecutive calendar days with the lowest average electricity price
# - then optimize 7 circles of 24 process-hours
# - each circle may spill into the next calendar day
N_SELECTED_DAYS = 8
N_CIRCLES = 7

PV_EFFICIENCY = 0.18
PV_AREA_M2 = 66700.0

BATTERY_ENERGY_MWH = 20.0
BATTERY_EFFICIENCY = 0.92
THERMAL_BUFFER_CAPACITY_MWH_TH = 30.0

BATTERY_INITIAL_SOC_FRAC = 0.50
THERMAL_INITIAL_SOC_FRAC = 0.50

RB = 0.14
RTH = 0.14

SCRAP_BASE = 0.021
SCRAP_TRANSITION_LOAD = 0.00015
SCRAP_TRANSITION_EAF = 0.0009
SCRAP_LOAD_UTIL = 0.0004
SCRAP_THERMAL_SHORTFALL = 0.0003
SCRAP_COST_FACTOR = 0.05

SOC_TOLERANCE = 0.50 # Increased to 50% tolerance for cyclic SOC constraints

MAX_GRID_IMPORT_MWH = 200.0 # Added maximum grid import capacity to prevent unboundedness

In [9]:
sequence = [
    ("Step 1", "EAF1.1", 1, 1, 0, 0, 72, 9.0, 0.000000),
    ("Step 1", "EAF1.2", 1, 1, 0, 0, 78, 9.3, 0.000000),
    ("Step 1", "EAF1.3", 1, 1, 0, 0, 80, 9.4, 0.000000),
    ("Step 1", "EAF1.4", 1, 1, 0, 0, 72, 9.0, 0.825104),

    ("Step 2", "Secondary Downstream 1.1", 0, 0, 1, 0, 32, 1.0, 7.219660),
    ("Step 2", "Secondary Downstream 1.2", 0, 0, 1, 0, 26, 0.7, 30.941400),
    ("Step 2", "Secondary Downstream 1.3", 0, 0, 1, 0, 20, 0.4, 22.690360),

    ("Step 3", "SD sequence Blue Colour", 0, 0, 1, 0, 15, 0.15, 1.340794),

    ("Step 4", "Minimum Critical Load 1", 0, 0, 0, 12, 12, 0.0, 1.134518),
    ("Step 4", "Minimum Critical Load 2", 0, 0, 0, 12, 12, 0.0, 1.031380),
    ("Step 4", "Minimum Critical Load 3", 0, 0, 0, 12, 12, 0.0, 0.618828),
    ("Step 4", "Minimum Critical Load 4", 0, 0, 0, 12, 12, 0.0, 0.618828),
    ("Step 4", "Minimum Critical Load 5", 0, 0, 0, 12, 12, 0.0, 0.618828),
    ("Step 4", "Minimum Critical Load 6", 0, 0, 0, 12, 12, 0.0, 2.887864),

    ("Step 5", "Preparation Blue 1", 0, 0, 1, 0, 15, 0.15, 2.062760),
    ("Step 5", "Preparation Blue 2", 0, 0, 1, 0, 24, 0.60, 0.515690),
    ("Step 5", "Preparation Blue 3", 0, 0, 1, 0, 38, 1.30, 4.125520),

    ("Step 6", "EAF2.1", 1, 1, 0, 0, 72, 9.0, 0.000000),
    ("Step 6", "EAF2.2", 1, 1, 0, 0, 78, 9.3, 0.412552),
    ("Step 6", "EAF2.3", 1, 1, 0, 0, 80, 9.4, 0.000000),
    ("Step 6", "EAF2.4", 1, 1, 0, 0, 76, 9.2, 0.825104),

    ("Step 7", "Secondary Downstream 2.1", 0, 0, 1, 0, 28, 0.8, 32.179056),
    ("Step 7", "Secondary Downstream 2.2", 0, 0, 1, 0, 24, 0.6, 30.941400),
    ("Step 7", "Secondary Downstream 2.3", 0, 0, 1, 0, 20, 0.4, 15.470700),
]

SEQ_DF = pd.DataFrame(sequence, columns=[
    "Step", "Production Phase", "ΕAF", "EAF,binary", "2ndary", "critical load MW",
    "Τotal Load", "Recovered Heat (MWh)", "Heat Required (MWh)"
])
SEQ_DF["t_in_day"] = np.arange(1, 25)

SEQ_DF.head()

,Step,Production Phase,ΕAF,"EAF,binary",2ndary,critical load MW,Τotal Load,Recovered Heat (MWh),Heat Required (MWh),t_in_day
0,Step 1,EAF1.1,1,1,0,0,72,9.0,0.000000,1
1,Step 1,EAF1.2,1,1,0,0,78,9.3,0.000000,2
2,Step 1,EAF1.3,1,1,0,0,80,9.4,0.000000,3
3,Step 1,EAF1.4,1,1,0,0,72,9.0,0.825104,4
4,Step 2,Secondary Downstream 1.1,0,0,1,0,32,1.0,7.219660,5


In [13]:
df = pd.read_excel(INPUT_FILE)
print("Original columns in Excel file:", df.columns.tolist()) # Added for debugging

df = df.rename(columns={
    "Electricity Price (€/MWh)": "Electricity Price",
    "Timestamp": "Datetime",
    "Natural Gas Price (€/MWh)": "Natural Gas Price",
    "Solar_Availability (kW/m^2)": "Solar Availability" # Corrected based on standard_output
})

required_cols = ["Datetime", "Electricity Price", "Natural Gas Price", "Solar Availability"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["Datetime"] = pd.to_datetime(df["Datetime"])
df["Electricity Price"] = pd.to_numeric(df["Electricity Price"], errors="coerce")
df["Natural Gas Price"] = pd.to_numeric(df["Natural Gas Price"], errors="coerce")
df["Solar Availability"] = pd.to_numeric(df["Solar Availability"], errors="coerce")

df = df.dropna(subset=["Datetime", "Electricity Price"]).copy()
df["Date"] = df["Datetime"].dt.date
df["Hour"] = df["Datetime"].dt.hour

# keep only complete 24-hour days
counts = df.groupby("Date").size()
complete_days = counts[counts == 24].index
df = df[df["Date"].isin(complete_days)].copy().sort_values("Datetime").reset_index(drop=True)

df.head()

Original columns in Excel file: ['Electricity Price (€/MWh)', 'Timestamp', 'Hour', 'Natural Gas Price (€/MWh)', 'Solar_Availability (kW/m^2)']


,Electricity Price,Datetime,Hour,Natural Gas Price,Solar Availability,Date
0,134.06,2025-01-01 00:00:00,0,45.3,0.0,2025-01-01
1,124.42,2025-01-01 01:00:00,1,45.3,0.0,2025-01-01
2,118.60,2025-01-01 02:00:00,2,45.3,0.0,2025-01-01
3,112.38,2025-01-01 03:00:00,3,45.3,0.0,2025-01-01
4,108.00,2025-01-01 04:00:00,4,45.3,0.0,2025-01-01


In [14]:
daily_avg = df.groupby("Date", as_index=False)["Electricity Price"].mean()
daily_avg["Date"] = pd.to_datetime(daily_avg["Date"])
daily_avg = daily_avg.sort_values("Date").reset_index(drop=True)

best_avg = None
best_start = None
window_rows = []

for i in range(len(daily_avg) - N_SELECTED_DAYS + 1):
    w = daily_avg.iloc[i:i+N_SELECTED_DAYS].copy()
    start = w["Date"].iloc[0]
    expected = pd.date_range(start=start, periods=N_SELECTED_DAYS, freq="D")
    if list(w["Date"]) != list(expected):
        continue

    avg_val = w["Electricity Price"].mean()
    window_rows.append({
        "Window Start": w["Date"].iloc[0],
        "Window End": w["Date"].iloc[-1],
        "Average Electricity Price Over 8 Days": avg_val
    })

    if best_avg is None or avg_val < best_avg:
        best_avg = avg_val
        best_start = w["Date"].iloc[0]

if best_start is None:
    raise ValueError("No valid 8-day consecutive window found.")

window_end = best_start + pd.Timedelta(days=N_SELECTED_DAYS - 1)

df_window_ranking = pd.DataFrame(window_rows).sort_values(
    "Average Electricity Price Over 8 Days", ascending=True
).reset_index(drop=True)

selected_dates_8 = list(pd.date_range(best_start, periods=N_SELECTED_DAYS, freq="D").date)
selected_dates_7 = selected_dates_8[:N_CIRCLES]

selected_8d = df[df["Date"].isin(selected_dates_8)].copy().sort_values("Datetime").reset_index(drop=True)

# Add one-day buffer after day 7 for spillover
buffer_end_date = pd.Timestamp(selected_dates_7[-1]) + pd.Timedelta(days=1)
selected_with_buffer = df[
    (pd.to_datetime(df["Date"]) >= pd.Timestamp(selected_dates_7[0])) &
    (pd.to_datetime(df["Date"]) <= buffer_end_date)
].copy().sort_values("Datetime").reset_index(drop=True)

print("Lowest 8-day window:", best_start, "to", window_end)
print("Average electricity price:", round(best_avg, 4))
df_window_ranking.head()

Lowest 8-day window: 2025-04-27 00:00:00 to 2025-05-04 00:00:00
Average electricity price: 54.0977


,Window Start,Window End,Average Electricity Price Over 8 Days
0,2025-04-27,2025-05-04,54.097656
1,2025-04-26,2025-05-03,57.139740
2,2025-04-28,2025-05-05,59.525365
3,2025-04-25,2025-05-02,61.151406
4,2025-05-25,2025-06-01,62.190833


In [15]:
def pv_generation(solar_kw_m2):
    return (solar_kw_m2 * PV_EFFICIENCY * PV_AREA_M2) / 1000.0

# map each selected start-day to the first row index in buffered chronology
date_to_daystart = {}
for d in selected_dates_7:
    day_slice = selected_with_buffer[selected_with_buffer["Date"] == d]
    if day_slice.empty:
        raise ValueError(f"No data for selected day {d}")
    date_to_daystart[d] = day_slice.index.min()

date_to_daystart

{datetime.date(2025, 4, 27): np.int64(0),
 datetime.date(2025, 4, 28): np.int64(24),
 datetime.date(2025, 4, 29): np.int64(48),
 datetime.date(2025, 4, 30): np.int64(72),
 datetime.date(2025, 5, 1): np.int64(96),
 datetime.date(2025, 5, 2): np.int64(120),
 datetime.date(2025, 5, 3): np.int64(144)}

In [44]:
import os
import pandas as pd
import numpy as np
from pyomo.environ import (
    ConcreteModel, Var, Binary, NonNegativeReals, RangeSet,
    Constraint, Objective, minimize, SolverFactory, value
)
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
from pyomo.opt import SolverStatus, TerminationCondition

def solve_one_circle(base):
    m = ConcreteModel()
    m.T = RangeSet(0, 23)

    # Binary variables
    m.bc = Var(m.T, domain=Binary)   # Battery_charge* (binary decision to charge)
    m.bd = Var(m.T, domain=Binary)   # Battery_discharge* (binary decision to discharge)
    m.td = Var(m.T, domain=Binary)   # Thermal_discharge* (binary decision to discharge thermal buffer)

    # State / continuous variables
    m.soc = Var(m.T, domain=NonNegativeReals, bounds=(0, BATTERY_ENERGY_MWH)) # Battery State of Charge
    m.tsoc = Var(m.T, domain=NonNegativeReals, bounds=(0, THERMAL_BUFFER_CAPACITY_MWH_TH)) # Thermal buffer State of Charge
    m.grid = Var(m.T, domain=NonNegativeReals) # Grid import
    m.thermal_short = Var(m.T, domain=NonNegativeReals) # Thermal energy shortfall
    m.scrap = Var(m.T, domain=NonNegativeReals) # Scrap rate
    m.thermal_spill = Var(m.T, domain=NonNegativeReals) # Excess recovered heat that cannot be stored
    m.thermal_underflow = Var(m.T, domain=NonNegativeReals) # Thermal deficit that cannot be covered

    # Auxiliary continuous variables for actual charge/discharge amounts
    charge_max_rate = BATTERY_ENERGY_MWH * RB * BATTERY_EFFICIENCY
    discharge_max_rate = BATTERY_ENERGY_MWH * RB / BATTERY_EFFICIENCY
    m.charge_actual = Var(m.T, domain=NonNegativeReals, bounds=(0, charge_max_rate))
    m.discharge_actual = Var(m.T, domain=NonNegativeReals, bounds=(0, discharge_max_rate))

    # No simultaneous battery charge and discharge
    def no_sim_rule(m, t):
        return m.bc[t] + m.bd[t] <= 1
    m.no_sim = Constraint(m.T, rule=no_sim_rule)

    # Daily counts (Re-enabling as they were not the cause of infeasibility)
    m.charge_cnt = Constraint(expr=sum(m.bc[t] for t in m.T) >= 4)
    m.discharge_cnt = Constraint(expr=sum(m.bd[t] for t in m.T) <= 3)
    m.thermal_cnt = Constraint(expr=sum(m.td[t] for t in m.T) <= 8)

    # Battery SOC
    init_soc = BATTERY_ENERGY_MWH * BATTERY_INITIAL_SOC_FRAC

    def soc_balance_rule(m, t):
        if t == 0:
            return m.soc[t] == init_soc
        return m.soc[t] == m.soc[t-1] + m.charge_actual[t] - m.discharge_actual[t]
    m.soc_balance_con = Constraint(m.T, rule=soc_balance_rule)

    # Link binary variables to actual charge/discharge amounts and SOC limits
    def charge_limit_actual_upper(m, t):
        return m.charge_actual[t] <= charge_max_rate * m.bc[t]
    m.charge_limit_actual_upper = Constraint(m.T, rule=charge_limit_actual_upper)

    def charge_limit_soc_space(m, t):
        if t == 0:
            return m.charge_actual[t] <= BATTERY_ENERGY_MWH - init_soc
        return m.charge_actual[t] <= BATTERY_ENERGY_MWH - m.soc[t-1]
    m.charge_limit_soc_space = Constraint(m.T, rule=charge_limit_soc_space)

    def discharge_limit_actual_upper(m, t):
        return m.discharge_actual[t] <= discharge_max_rate * m.bd[t]
    m.discharge_limit_actual_upper = Constraint(m.T, rule=discharge_limit_actual_upper)

    def discharge_limit_soc_energy(m, t):
        if t == 0:
            return m.discharge_actual[t] <= init_soc
        return m.discharge_actual[t] <= m.soc[t-1]
    m.discharge_limit_soc_energy = Constraint(m.T, rule=discharge_limit_soc_energy)

    # Temporarily commenting out cyclic SOC constraints to address infeasibility
    # m.soc_cycle_lower = Constraint(expr=m.soc[23] >= m.soc[0] * (1 - SOC_TOLERANCE))
    # m.soc_cycle_upper = Constraint(expr=m.soc[23] <= m.soc[0] * (1 + SOC_TOLERANCE))

    # Thermal SOC initialisation and balance
    init_tsoc = THERMAL_BUFFER_CAPACITY_MWH_TH * THERMAL_INITIAL_SOC_FRAC

    # Explicitly fix initial values for t=0
    m.tsoc[0].fix(init_tsoc)
    m.thermal_spill[0].fix(0.0) # No spill at t=0 before process starts
    m.thermal_underflow[0].fix(0.0) # No underflow at t=0 before process starts

    # Thermal SOC balance for t > 0
    m.T_balance = RangeSet(1, 23)
    def tsoc_balance_rule(m, t):
        rec = float(base.loc[t, "Recovered Heat (MWh)"])
        thermal_discharge_amt_hr = THERMAL_BUFFER_CAPACITY_MWH_TH * RTH # Corrected rate for consistency
        return m.tsoc[t] == m.tsoc[t-1] + rec - thermal_discharge_amt_hr * m.td[t] - m.thermal_spill[t] + m.thermal_underflow[t]
    m.tsoc_con = Constraint(m.T_balance, rule=tsoc_balance_rule)

    # Temporarily commenting out cyclic Thermal SOC constraints to address infeasibility
    # m.tsoc_cycle_lower = Constraint(expr=m.tsoc[23] >= m.tsoc[0] * (1 - SOC_TOLERANCE))
    # m.tsoc_cycle_upper = Constraint(expr=m.tsoc[23] <= m.tsoc[0] * (1 + SOC_TOLERANCE))

    # Grid import
    def grid_rule(m, t):
        load = float(base.loc[t, "Τotal Load"])
        pv = float(base.loc[t, "PV Generation MWh"])
        return m.grid[t] >= load - pv + m.charge_actual[t] - m.discharge_actual[t] # Use actual charge/discharge
    m.grid_con = Constraint(m.T, rule=grid_rule)

    # Add upper bound for grid import to prevent unboundedness with negative prices
    def grid_upper_bound_rule(m, t):
        return m.grid[t] <= MAX_GRID_IMPORT_MWH
    m.grid_upper_con = Constraint(m.T, rule=grid_upper_bound_rule)

    # Thermal shortfall
    def thermal_short_rule(m, t):
        heat_req = float(base.loc[t, "Heat Required (MWh)"])
        rec = float(base.loc[t, "Recovered Heat (MWh)"])
        thermal_discharge_amt_hr = THERMAL_BUFFER_CAPACITY_MWH_TH * RTH # Corrected rate for consistency
        return m.thermal_short[t] >= heat_req - rec - thermal_discharge_amt_hr * m.td[t]
    m.thermal_short_con = Constraint(m.T, rule=thermal_short_rule)

    # Scrap rate linear lower bound
    max_load = float(SEQ_DF["Τotal Load"].max())

    def scrap_rule(m, t):
        load_t = float(base.loc[t, "Τotal Load"])
        eaf_t = float(base.loc[t, "EAF,binary"])
        if t == 0:
            prev_load = float(base.loc[0, "Τotal Load"])
            prev_eaf = float(base.loc[0, "EAF,binary"])
        else:
            prev_load = float(base.loc[t-1, "Τotal Load"])
            prev_eaf = float(base.loc[t-1, "EAF,binary"])

        abs_load_diff = abs(load_t - prev_load)
        abs_eaf_diff = abs(eaf_t - prev_eaf)

        rhs = (
            SCRAP_BASE
            + SCRAP_TRANSITION_LOAD * abs_load_diff
            + SCRAP_TRANSITION_EAF * abs_eaf_diff
            + SCRAP_LOAD_UTIL * (1 - prev_load / max_load)
            + SCRAP_THERMAL_SHORTFALL * m.thermal_short[t]
        )
        return m.scrap[t] >= rhs
    m.scrap_con = Constraint(m.T, rule=scrap_rule)

    # Objective
    def obj_rule(m):
        total = 0
        for t in m.T:
            elec = float(base.loc[t, "Electricity Price"])
            gas = float(base.loc[t, "Natural Gas Price"]) if pd.notna(base.loc[t, "Natural Gas Price"]) else 0.0
            load = float(base.loc[t, "Τotal Load"])
            total += (
                m.grid[t] * elec
                + m.thermal_short[t] * gas
                + m.scrap[t] * load * SCRAP_COST_FACTOR
                + m.thermal_underflow[t] * gas # Penalize thermal underflow with gas price
            )
        return total
    m.obj = Objective(rule=obj_rule, sense=minimize)

    solver = SolverFactory("glpk")
    if not solver.available(False):
        raise RuntimeError("GLPK solver is not available.")

    result = solver.solve(m, tee=False)

    # Updated robust check for solution quality
    if (result.solver.status == SolverStatus.ok and
        (result.solver.termination_condition == TerminationCondition.optimal or
         result.solver.termination_condition == TerminationCondition.feasible)):
        # Solution found, proceed to extract values
        pass
    else:
        # Solver did not find an acceptable solution, return None
        print(f"Solver did not find an optimal or feasible solution for this circle. Status: {result.solver.status}, Termination: {result.solver.termination_condition}")
        return None, None

    out = base.copy()
    out["Battery_charge*"] = [int(round(value(m.bc[t]))) for t in m.T]
    out["Battery_discharge*"] = [int(round(value(m.bd[t]))) for t in m.T]
    out["Thermal_discharge*"] = [int(round(value(m.td[t]))) for t in m.T]
    out["Battery SOC (MWh)"] = [value(m.soc[t]) for t in m.T]
    out["Thermal_SOC"] = [value(m.tsoc[t]) for t in m.T]
    out["Grid_import"] = [value(m.grid[t]) for t in m.T]
    out["Scrap_rate"] = [value(m.scrap[t]) for t in m.T]
    out["Thermal_spill"] = [value(m.thermal_spill[t]) for t in m.T] # Add thermal spill to output
    out["Thermal_underflow"] = [value(m.thermal_underflow[t]) for t in m.T] # Add thermal underflow to output
    out["Battery_charge_actual"] = [value(m.charge_actual[t]) for t in m.T] # Add actual charge output
    out["Battery_discharge_actual"] = [value(m.discharge_actual[t]) for t in m.T] # Add actual discharge output

    out["Total Cost without PV and Heat Recover"] = (
        (out["ΕAF"] + out["2ndary"] + out["critical load MW"]) * out["Electricity Price"]
    )
    out["Total Cost with PV and Storage"] = out["Grid_import"] * out["Electricity Price"]

    thermal_short = np.array([value(m.thermal_short[t]) for t in m.T])
    out["Total Cost with PV+Storage+Scrap+Thermal"] = (
        out["Grid_import"] * out["Electricity Price"]
        + thermal_short * out["Natural Gas Price"].fillna(0)
        + out["Scrap_rate"] * out["Τotal Load"] * SCRAP_COST_FACTOR
        + out["Thermal_underflow"] * out["Natural Gas Price"].fillna(0) # Added cost for thermal underflow
    )

    total_cost = float(out["Total Cost with PV+Storage+Scrap+Thermal"].sum())
    return out, total_cost

In [45]:
all_results = []
daily_summary = []
global_t_counter = 1

for circle_idx, day in enumerate(selected_dates_7, start=1):
    day_start_idx = date_to_daystart[day]

    # ------------------------------------------
    # SMART START SELECTION (NO LOOP)
    # choose best 4-hour cheap block directly
    # ------------------------------------------
    best_h = None
    best_cost = None

    for h in range(24):
        start_idx = day_start_idx + h
        end_idx = start_idx + 3

        if end_idx >= len(selected_with_buffer):
            continue

        block = selected_with_buffer.iloc[start_idx:start_idx+4]

        cost = block["Electricity Price"].sum()

        if best_cost is None or cost < best_cost:
            best_cost = cost
            best_h = h

    print(f"Circle {circle_idx} | Start hour selected directly: {best_h}")

    # ------------------------------------------
    # Build ONLY ONE MILP per day
    # ------------------------------------------
    start_idx = day_start_idx + best_h
    chrono = selected_with_buffer.iloc[start_idx:start_idx+24].copy().reset_index(drop=True)

    base = chrono.copy()
    base["t_in_day"] = np.arange(1, 25)
    base["Step"] = SEQ_DF["Step"].values
    base["Production Phase"] = SEQ_DF["Production Phase"].values
    base["ΕAF"] = SEQ_DF["ΕAF"].values
    base["EAF,binary"] = SEQ_DF["EAF,binary"].values
    base["2ndary"] = SEQ_DF["2ndary"].values
    base["critical load MW"] = SEQ_DF["critical load MW"].values
    base["Τotal Load"] = SEQ_DF["Τotal Load"].values
    base["Recovered Heat (MWh)"] = SEQ_DF["Recovered Heat (MWh)"].values
    base["Heat Required (MWh)"] = SEQ_DF["Heat Required (MWh)"].values
    base["Battery energy MWh"] = BATTERY_ENERGY_MWH
    base["Battery efficiency (Fraction)"] = BATTERY_EFFICIENCY
    base["Thermal buffer capacity MWh_th"] = THERMAL_BUFFER_CAPACITY_MWH_TH
    base["PV Generation MWh"] = base["Solar Availability"].fillna(0).apply(pv_generation)
    base["Battery Charge/Disch Rate Rb (fraction)"] = RB
    base["Thermal Charge/Disch Rate Rth (fraction)"] = RTH

    solved_df, solved_cost = solve_one_circle(base)

    if solved_df is None:
        raise RuntimeError(f"Solver failed for day {day}")

    daily_summary.append({
        "Circle": circle_idx,
        "Date": pd.Timestamp(day),
        "Start Hour (EAF1.1)": best_h,
        "Daily Cost": solved_cost
    })

    for _, row in solved_df.iterrows():
        rec = row.to_dict()
        rec["Global t"] = global_t_counter
        rec["Date"] = pd.Timestamp(day)
        all_results.append(rec)
        global_t_counter += 1

df_daily = pd.DataFrame(daily_summary)
df_results = pd.DataFrame(all_results)

print("DONE ✅")
df_daily

Circle 1 | Start hour selected directly: 9
Circle 2 | Start hour selected directly: 9
Circle 3 | Start hour selected directly: 9
Circle 4 | Start hour selected directly: 10
Circle 5 | Start hour selected directly: 10
Circle 6 | Start hour selected directly: 9
Circle 7 | Start hour selected directly: 9
DONE ✅


,Circle,Date,Start Hour (EAF1.1),Daily Cost
0,1,2025-04-27,9,53861.590491
1,2,2025-04-28,9,43775.494917
2,3,2025-04-29,9,43999.156804
3,4,2025-04-30,10,31365.534935
4,5,2025-05-01,10,6311.236969
5,6,2025-05-02,9,46085.764080
6,7,2025-05-03,9,19407.598896


In [46]:
ordered_cols = [
    "Global t", "Date", "t_in_day", "Datetime", "Hour",
    "Step", "Production Phase",
    "Electricity Price", "Natural Gas Price", "Solar Availability",
    "Battery energy MWh", "Battery efficiency (Fraction)", "Thermal buffer capacity MWh_th",
    "PV Generation MWh", "ΕAF", "EAF,binary", "2ndary", "critical load MW",
    "Τotal Load", "Recovered Heat (MWh)", "Heat Required (MWh)",
    "Battery_charge*", "Battery_discharge*", "Thermal_discharge*",
    "Battery Charge/Disch Rate Rb (fraction)", "Battery SOC (MWh)",
    "Thermal Charge/Disch Rate Rth (fraction)", "Thermal_SOC",
    "Grid_import", "Scrap_rate", "Thermal_spill", "Thermal_underflow", # Added Thermal_underflow
    "Battery_charge_actual", "Battery_discharge_actual", # Added new actual charge/discharge cols
    "Total Cost without PV and Heat Recover",
    "Total Cost with PV and Storage",
    "Total Cost with PV+Storage+Scrap+Thermal"
]

for c in ordered_cols:
    if c not in df_results.columns:
        df_results[c] = np.nan

df_results = df_results[ordered_cols]

df_summary = pd.DataFrame({
    "Metric": [
        "Selected 8-day window start",
        "Selected 8-day window end",
        "Lowest 8-day average electricity price",
        "Optimization circles used",
        "Total process times",
        "Total optimized cost with PV+Storage+Scrap+Thermal"
    ],
    "Value": [
        best_start,
        window_end,
        best_avg,
        N_CIRCLES,
        24 * N_CIRCLES,
        # Check if df_results is not empty before summing
        float(df_results["Total Cost with PV+Storage+Scrap+Thermal"].sum()) if not df_results.empty else 0.0
    ]
})

df_notes = pd.DataFrame({
    "Notes": [
        "This file was produced by the GLPK-ready Pyomo model.",
        "The 8 lowest-average consecutive calendar days were selected first.",
        "The first 7 selected calendar days were used as the 7 process circles.",
        "Each circle can spill into the next calendar day because its 24 process-hours are mapped chronologically from the optimized start hour.",
        "Battery_charge*, Battery_discharge*, Thermal_discharge* are solver-based binary decisions.",
        "Daily constraints enforced: sum(Battery_charge*) >= 4, sum(Battery_discharge*) <= 3, sum(Thermal_discharge*) <= 8.",
        "Cyclic closure enforced: Battery SOC at t=24 is within 50% of Battery SOC at t=1, and same for Thermal_SOC. (Currently commented out due to infeasibility challenges.)",
        "Introduced 'Thermal_spill' variable to handle excess recovered heat that cannot be stored in the thermal buffer.",
        "Introduced 'Thermal_underflow' variable to handle thermal deficits and ensure feasibility of the thermal buffer balance, penalised by gas price.",
        "Battery charge/discharge amounts (`Battery_charge_actual`, `Battery_discharge_actual`) are now explicitly limited by available SOC and capacity, linked to binary decisions."
    ]
})

df_summary

,Metric,Value
0,Selected 8-day window start,2025-04-27 00:00:00
1,Selected 8-day window end,2025-05-04 00:00:00
2,Lowest 8-day average electricity price,54.097656
3,Optimization circles used,7
4,Total process times,168
5,Total optimized cost with PV+Storage+Scrap+The...,244806.377092


In [47]:
with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    df_summary.to_excel(writer, sheet_name="Summary", index=False)
    df_window_ranking.to_excel(writer, sheet_name="8-Day Window Ranking", index=False)
    selected_8d[["Datetime", "Date", "Hour", "Electricity Price", "Natural Gas Price", "Solar Availability"]].to_excel(
        writer, sheet_name="Selected Prices", index=False
    )
    df_daily.to_excel(writer, sheet_name="Daily Optimization", index=False)
    df_results.to_excel(writer, sheet_name="Optimized Results", index=False)
    df_notes.to_excel(writer, sheet_name="Notes", index=False)

print("Excel written:", OUTPUT_FILE)

Excel written: MiniMill_Design_Formula_Integrated_Output_GLPK.xlsx


In [48]:
wb = load_workbook(OUTPUT_FILE)

header_fill = PatternFill("solid", fgColor="1F4E78")
header_font = Font(color="FFFFFF", bold=True)

for ws in wb.worksheets:
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    for col_cells in ws.columns:
        col_letter = get_column_letter(col_cells[0].column)
        max_len = 0
        for cell in col_cells[:200]:
            val = "" if cell.value is None else str(cell.value)
            max_len = max(max_len, len(val))
        ws.column_dimensions[col_letter].width = min(max_len + 2, 28)

wb.save(OUTPUT_FILE)
print("Workbook formatted:", OUTPUT_FILE)

Workbook formatted: MiniMill_Design_Formula_Integrated_Output_GLPK.xlsx


In [49]:
from google.colab import files
files.download(OUTPUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [50]:
print("Selected 8-day window:", best_start, "to", window_end)
print("Lowest 8-day average electricity price:", round(best_avg, 4))
print("Optimization circles used:", N_CIRCLES)
print("Total optimized cost with PV+Storage+Scrap+Thermal:",
      round(float(df_results["Total Cost with PV+Storage+Scrap+Thermal"].sum()), 2))

df_daily

Selected 8-day window: 2025-04-27 00:00:00 to 2025-05-04 00:00:00
Lowest 8-day average electricity price: 54.0977
Optimization circles used: 7
Total optimized cost with PV+Storage+Scrap+Thermal: 244806.38


,Circle,Date,Start Hour (EAF1.1),Daily Cost
0,1,2025-04-27,9,53861.590491
1,2,2025-04-28,9,43775.494917
2,3,2025-04-29,9,43999.156804
3,4,2025-04-30,10,31365.534935
4,5,2025-05-01,10,6311.236969
5,6,2025-05-02,9,46085.764080
6,7,2025-05-03,9,19407.598896
